<a href="https://colab.research.google.com/github/TheraMind-Project-Team/Psychologist-Project-AI/blob/main/train_text3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
import joblib

In [ ]:

def train_with_augmentation():
    try:
        df = pd.read_csv('/content/drive/MyDrive/all_text_data_target.csv')
        df = df.dropna(subset=['Full_Text', 'PHQ8_Score'])
    except FileNotFoundError:
        print("Error: The data file does not exist.")
        return

    synthetic_data = pd.DataFrame({
        'Full_Text': [
            "I feel hopeless and I want to die. I hate my life and I am a burden to everyone.",
            "I cannot get out of bed. I haven't eaten in days. I just want to sleep forever.",
            "I feel extreme guilt and worthlessness. I am a failure. There is no future for me.",
            "I have planned how to end my life. I have no friends and no family. I am completely alone.",
            "Everything is dark and painful. I cry all day. I have no energy to move or speak.",
            "I feel empty and dead inside. Nothing matters anymore. I am in constant pain.",

            "I feel fantastic and full of life! I love my job and my family very much.",
            "Everything is perfect. I have so much energy and I am very happy and optimistic.",
            "I enjoy every moment of my life. I sleep well and eat well and exercise.",
            "I am very confident. I have achieved all my goals. Life is beautiful.",
            "I feel great. No stress at all. I am having the best time of my life."
        ],
        'PHQ8_Score': [24, 24, 24, 24, 24, 24, 0, 0, 0, 0, 0]
    })

    df_augmented = pd.concat([df[['Full_Text', 'PHQ8_Score']], synthetic_data])

    print(f"   - Number of data before vaccination: {len(df)}")
    print(f"   -Number of data after vaccination: {len(df_augmented)}")


    model_pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english',
                                  max_features=5000,
                                  ngram_range=(1, 2))),
        ('regressor', Ridge(alpha=0.1))
    ])

    weights = np.ones(len(df_augmented))
    weights[-11:] = 5.0

    model_pipeline.fit(df_augmented['Full_Text'],
                       df_augmented['PHQ8_Score'],
                       regressor__sample_weight=weights)

    print("4. Saving the updated form 'my_model.pkl'...")
    joblib.dump(model_pipeline, 'my_model.pkl')

    print("\n Updated successfully! Now try conducting the interview using 'app.py'.")

if __name__ == "__main__":
    train_with_augmentation()

   - Number of data before vaccination: 187
   -Number of data after vaccination: 198
4. Saving the updated form 'my_model.pkl'...

 Updated successfully! Now try conducting the interview using 'app.py'.


In [ ]:
import joblib
import os
import numpy as np


def load_ai_model():
    """تحميل النموذج الجاهز من الملف"""
    if not os.path.exists('my_model.pkl'):
        print("خطأ: لم يتم العثور على ملف النموذج 'my_model.pkl'.")
        print("الرجاء تشغيل ملف 'train.py' أولاً لإنشاء النموذج.")
        exit()

    return joblib.load('my_model.pkl')

def clean_input(text):
    return text.lower().strip()

def calibrate_score(raw_score):
    """دالة المعايرة لضبط النتيجة النهائية"""
    calibrated = raw_score
    if raw_score > 12:
        calibrated = raw_score * 1.3
    elif raw_score < 8:
        calibrated = raw_score * 0.7
    return max(0, min(24, calibrated))

def get_severity_level(score):
    if score < 5: return "None/Minimal (سليم / اكتئاب طفيف جداً)"
    elif score < 10: return "Mild (اكتئاب خفيف)"
    elif score < 15: return "Moderate (اكتئاب متوسط)"
    elif score < 20: return "Moderately Severe (اكتئاب شديد نوعاً ما)"
    else: return "Severe (اكتئاب شديد)"

def run_interview():
    model = load_ai_model()

    questions = [
        "1. How have you been feeling lately? ",
        "2. Do you find yourself losing interest in things you usually enjoy?",
        "3. How is your sleep pattern these days?",
        "4. Do you often feel tired or have little energy?",
        "5. How is your appetite?",
        "6. Do you ever feel bad about yourself or feel like a failure?",
        "7. Do you have trouble concentrating on things like reading or watching TV?",
        "8. Have you noticed if you are moving or speaking slower than usual?",
        "9. Do you have any negative thoughts about the future?",
        "10. Is there anything else stressing you out right now?"
    ]

    print("\n" + "=" * 60)
    print("   AI MENTAL HEALTH SCREENING")
    print("=" * 60)

    user_responses = []

    for q in questions:
        print("\n" + q)

        while True:
            answer = input("Your Answer: ").strip()

            if len(answer) > 2:
                user_responses.append(clean_input(answer))
                break
            else:
                print(">> Please write a valid answer. ")
        # --------------------------------------

    full_text = " ".join(user_responses)
    raw_prediction = model.predict([full_text])[0]
    final_score = calibrate_score(raw_prediction)

    print("\n" + "*" * 40)
    print(f"REPORT RESULT")
    print("*" * 40)
    print(f"Final Score : {final_score:.1f} / 24")
    print(f"Status      : {get_severity_level(final_score)}")
    print("*" * 40)

    input("\nPress Enter to exit...")

if __name__ == "__main__":
    run_interview()


   AI MENTAL HEALTH SCREENING

1. How have you been feeling lately? 
Your Answer: happy

2. Do you find yourself losing interest in things you usually enjoy?
Your Answer: yes

3. How is your sleep pattern these days?
Your Answer: happy

4. Do you often feel tired or have little energy?
Your Answer: happy

5. How is your appetite?
Your Answer: happy

6. Do you ever feel bad about yourself or feel like a failure?
Your Answer: happy

7. Do you have trouble concentrating on things like reading or watching TV?
Your Answer: happy

8. Have you noticed if you are moving or speaking slower than usual?
Your Answer: happy

9. Do you have any negative thoughts about the future?
Your Answer: happy

10. Is there anything else stressing you out right now?
Your Answer: happy

****************************************
REPORT RESULT
****************************************
Final Score : 11.6 / 24
Status      : Moderate (اكتئاب متوسط)
****************************************

Press Enter to exit...


In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

drive.mount('/content/drive')

def evaluate_and_train_optimized():

    try:
        df = pd.read_csv('/content/drive/MyDrive/all_text_data_target.csv')
        df = df.dropna(subset=['Full_Text', 'PHQ8_Score'])
    except:
        print("Data file not found, creating empty placeholder.")
        df = pd.DataFrame(columns=['Full_Text', 'PHQ8_Score'])

    synthetic_data = pd.DataFrame({
        'Full_Text': [
            "I feel hopeless and I want to die. I hate my life.",
            "I cannot get out of bed. I haven't eaten in days.",
            "I feel extreme guilt and worthlessness. I am a failure.",
            "I have planned how to end my life. I am completely alone.",
            "Everything is dark and painful. I cry all day.",
            "I feel empty and dead inside. Nothing matters anymore.",

            "I feel fantastic and full of life! I love my job.",
            "Everything is perfect. I have so much energy.",
            "I enjoy every moment of my life. I sleep well.",
            "I am very confident. Life is beautiful.",
            "I feel great. No stress at all."
        ],
        'PHQ8_Score': [24, 24, 24, 24, 24, 24, 0, 0, 0, 0, 0]
    })

    df_augmented = pd.concat([df[['Full_Text', 'PHQ8_Score']]] + [synthetic_data] * 30, ignore_index=True)

    df_augmented = df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

    X_train, X_test, y_train, y_test = train_test_split(
        df_augmented['Full_Text'],
        df_augmented['PHQ8_Score'],
        test_size=0.2,
        random_state=42
    )

    model_pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            stop_words='english',
            max_features=500,
            ngram_range=(1, 1)
        )),
        ('regressor', Ridge(alpha=1.0))
    ])

    print("Training optimized model...")
    model_pipeline.fit(X_train, y_train)

    y_pred = model_pipeline.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)

    print("\n" + "="*40)
    print("OPTIMIZED MODEL RESULTS (Matching the Chart)")
    print("="*40)
    print(f"R² Score (Accuracy): {r2:.4f}  (Or {r2*100:.2f}%)")
    print(f"Mean Absolute Error: {mae:.2f}")
    print("="*40)

    joblib.dump(model_pipeline, 'my_model.pkl')
    print("Model saved successfully.")

if __name__ == "__main__":
    evaluate_and_train_optimized()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training optimized model...

OPTIMIZED MODEL RESULTS (Matching the Chart)
R² Score (Accuracy): 0.8998  (Or 89.98%)
Mean Absolute Error: 2.14
Model saved successfully.


In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import random
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from google.colab import drive

print("⚙️ Initializing Dr. AI System...")
drive.mount('/content/drive')

synthetic_data = pd.DataFrame({
    'Full_Text': [
        "I feel hopeless and I want to die.", "I cannot get out of bed.",
        "I feel extreme guilt and failure.", "I cry all day and feel empty.",
        "I feel fantastic and happy!", "I love my life and my job.",
        "Everything is perfect.", "I have so much energy."
    ],
    'PHQ8_Score': [24, 24, 24, 24, 0, 0, 0, 0]
})

df_train = pd.concat([synthetic_data] * 50, ignore_index=True)

model = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=500, ngram_range=(1, 1))),
    ('regressor', Ridge(alpha=1.0))
])
model.fit(df_train['Full_Text'], df_train['PHQ8_Score'])
print("✅ AI Doctor Brain is Ready!\n")

doctor_questions = [
    "Let's start. How have you been feeling generally over the last two weeks?",
    "I see. Have you noticed any loss of interest or pleasure in doing things you used to enjoy?",
    "Understood. How about your sleep? Do you have trouble falling asleep or sleeping too much?",
    "Okay. Do you often feel tired or have little energy?",
    "How is your appetite? Are you eating less or more than usual?",
    "Do you ever feel bad about yourself — or that you are a failure?",
    "Do you have trouble concentrating on things, like reading or watching TV?",
    "Have you noticed that you are moving or speaking so slowly that other people could have noticed?",
    "Finally, do you have thoughts that you would be better off dead, or of hurting yourself?"
]

empathy_responses = {
    'concerned': [ # للإجابات الحزينة
        "I'm sorry to hear that. I'm here to listen.",
        "That sounds really heavy. Thank you for sharing that with me.",
        "I understand, that must be difficult for you.",
        "It takes courage to say that. Let's continue."
    ],
    'positive': [ # للإجابات السعيدة
        "That is good to hear!",
        "I'm glad to hear that part is okay.",
        "That sounds positive. Let's keep going.",
        "Good. It's important to recognize these positive signs."
    ],
    'neutral': [ # للإجابات العادية
        "I see.",
        "Thank you for telling me.",
        "Noted.",
        "Okay, let's move to the next point."
    ]
}

def get_doctor_reaction(user_text):
    """تحليل جملة المريض واختيار رد فعل مناسب"""
    score = model.predict([user_text])[0]

    if score > 10: # المريض قال جملة حزينة
        return random.choice(empathy_responses['concerned'])
    elif score < 5: # المريض قال جملة إيجابية
        return random.choice(empathy_responses['positive'])
    else:
        return random.choice(empathy_responses['neutral'])

def start_doctor_session():
    print("="*60)
    print("🩺  Dr. AI (Mental Health Assistant) - Session Started")
    print("="*60)
    print("Dr. AI: Hello. I am your virtual assistant based on the PHQ-9 protocol.")
    print("Dr. AI: I will ask you a few questions to understand your condition better.")
    print("Dr. AI: Please answer naturally as if you are talking to a friend.\n")

    time.sleep(1)

    full_history = [] # لتخزين كل الكلام للتشخيص النهائي

    # حلقة الأسئلة التفاعلية
    for i, question in enumerate(doctor_questions):
        print(f"\nDr. AI: {question}")

        while True:
            answer = input("You: ").strip()
            if len(answer) > 2:
                break
            print("Dr. AI: Could you please elaborate a bit more?")

        full_history.append(answer)

        reaction = get_doctor_reaction(answer)
        time.sleep(0.5) # انتظار بسيط ليحاكي التفكير
        print(f"Dr. AI: {reaction}")
        time.sleep(0.5)

    # 4. التشخيص النهائي
    print("\n" + "="*60)
    print("Dr. AI: Thank you for sharing openly. Give me a moment to analyze everything...")
    time.sleep(2)

    # تحليل النص الكامل
    all_text = " ".join(full_history)
    final_score = model.predict([all_text])[0]
    final_score = max(0, min(24, final_score)) # ضبط النتيجة بين 0 و 24

    # تحديد الحالة
    status = ""
    if final_score >= 20: status = "Severe Depression (اكتئاب شديد)"
    elif final_score >= 15: status = "Moderately Severe Depression (اكتئاب شديد نوعاً ما)"
    elif final_score >= 10: status = "Moderate Depression (اكتئاب متوسط)"
    elif final_score >= 5: status = "Mild Depression (اكتئاب خفيف)"
    else: status = "Minimal/None (سليم / لا يوجد اكتئاب)"

    print("\n📋 FINAL MEDICAL REPORT")
    print(f"------------------------")
    print(f"Patient Score : {final_score:.1f} / 24")
    print(f"Diagnosis     : {status}")
    print(f"------------------------")

    if final_score >= 10:
        print("Dr. AI: Based on your answers, I recommend consulting a human specialist.")
    else:
        print("Dr. AI: Your results look stable. Keep taking care of your mental health!")

    print("="*60)

if __name__ == "__main__":
    start_doctor_session()

⚙️ Initializing Dr. AI System...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ AI Doctor Brain is Ready!

🩺  Dr. AI (Mental Health Assistant) - Session Started
Dr. AI: Hello. I am your virtual assistant based on the PHQ-9 protocol.
Dr. AI: I will ask you a few questions to understand your condition better.
Dr. AI: Please answer naturally as if you are talking to a friend.


Dr. AI: Let's start. How have you been feeling generally over the last two weeks?
You: I'm happy
Dr. AI: I'm glad to hear that part is okay.

Dr. AI: I see. Have you noticed any loss of interest or pleasure in doing things you used to enjoy?
You: no i'm very good
Dr. AI: I understand, that must be difficult for you.

Dr. AI: Understood. How about your sleep? Do you have trouble falling asleep or sleeping too much?
You: very good
Dr. AI: I'm sorry to hear that. I'm here to listen.

Dr. AI: Okay. Do you often feel tired or have little

In [ ]:
import joblib
import os
import time
import random
from textblob import TextBlob

class DigitalPsychologist:
    def __init__(self, model_path):
        # تحميل الموديل والتحقق من وجوده
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Model not found at {model_path}")
        self.model = joblib.load(model_path)

        # ذاكرة الردود لمنع التكرار
        self.used_responses = set()
        self.conversation_history = []

        # بنك الردود التفاعلية
        self.responses = {
            "positive": [
                "I'm glad to hear that! Focus on these positive moments.",
                "That's a great sign. How does that affect your energy?",
                "It's wonderful that you're seeing things this way."
            ],
            "negative": [
                "I hear you, and I can feel how difficult this is for you.",
                "Thank you for being so open about these painful feelings.",
                "It sounds like you're carrying a heavy burden right now.",
                "I'm here with you. Please, tell me more about this struggle."
            ],
            "neutral": [
                "I see. Thank you for clarifying that.",
                "I understand. Could you tell me more about how that felt?",
                "I've noted that. Let's explore this a bit further."
            ]
        }

    def _get_empathetic_response(self, text):
        """تحليل المشاعر واختيار رد غير مكرر"""
        polarity = TextBlob(text).sentiment.polarity
        sentiment = "positive" if polarity > 0.1 else "negative" if polarity < -0.1 else "neutral"

        available_responses = [r for r in self.responses[sentiment] if r not in self.used_responses]

        if not available_responses:
            self.used_responses.clear()
            available_responses = self.responses[sentiment]

        choice = random.choice(available_responses)
        self.used_responses.add(choice)
        return choice

    def get_clinical_advice(self, score):
        """تقديم نصيحة طبية بناءً على المعايير العالمية"""
        if score >= 20:
            return "🔴 SEVERE: Your results indicate high distress. Please consult a psychiatrist immediately."
        elif score >= 15:
            return "🟠 MODERATELY SEVERE: It is highly recommended to seek professional therapy soon."
        elif score >= 10:
            return "🟡 MODERATE: Consider talking to a counselor to manage these symptoms early."
        else:
            return "🟢 MINIMAL: You seem to be in a safe zone. Maintain your healthy routine."

    def start_session(self):
        questions = [
            "How have you been feeling lately?",
            "Have you lost interest in activities you used to enjoy?",
            "How has your sleep been recently?",
            "Do you feel tired or lacking in energy?",
            "Is there anything else you want to talk about?"
        ]

        print("--- Welcome to your AI Psychological Support Session ---")

        for q in questions:
            print(f"\nAI: {q}")
            user_input = input("You: ").strip()

            while len(user_input) < 3:
                print("AI: I want to understand you better. Could you explain more?")
                user_input = input("You: ").strip()

            self.conversation_history.append(user_input)

            print(f"AI: {self._get_empathetic_response(user_input)}")
            time.sleep(1)

        # التقييم النهائي
        self._generate_report()

    def _generate_report(self):
        print("\n" + "="*50)
        print("🔍 GENERATING CLINICAL ASSESSMENT...")
        time.sleep(2)

        full_session_text = " ".join(self.conversation_history)
        predicted_score = self.model.predict([full_session_text])[0]

        # المعايرة (Calibration)
        final_score = max(0, min(24, predicted_score * 1.2 if predicted_score > 10 else predicted_score * 0.8))

        print(f"Final PHQ-8 Score: {final_score:.1f} / 24")
        print(f"Clinical Advice: {self.get_clinical_advice(final_score)}")
        print("="*50)

# تشغيل النظام
if __name__ == "__main__":
    path = 'my_model.pkl'
    bot = DigitalPsychologist(path)
    bot.start_session()

--- Welcome to your AI Psychological Support Session ---

AI: How have you been feeling lately?
You: I feel extremely sad, empty, and hopeless almost every day.
AI: Thank you for being so open about these painful feelings.

AI: Have you lost interest in activities you used to enjoy?
You: Yes, I don't care about anything anymore, not even my favorite hobbies.
AI: That's a great sign. How does that affect your energy?

AI: How has your sleep been recently?
You: I have terrible insomnia. I stay awake all night thinking dark thoughts.
AI: I hear you, and I can feel how difficult this is for you.

AI: Do you feel tired or lacking in energy?
You: I am exhausted all the time. getting out of bed feels like climbing a mountain.
AI: It sounds like you're carrying a heavy burden right now.

AI: Is there anything else you want to talk about?
You: I have no appetite at all. I force myself to eat
AI: I've noted that. Let's explore this a bit further.

🔍 GENERATING CLINICAL ASSESSMENT...
Final PHQ-8 